# Régressions Linéaires et Régularisées sur la Production Minière

Ce notebook présente l'approche multivariée pour prédire la production en fonction des variables climatiques et économiques. Nous utilisons :

- **Régression linéaire (OLS)** comme référence,
- **Ridge Regression** avec validation croisée pour réduire le sur-ajustement,
- **Lasso Regression** pour réaliser une sélection automatique des variables.

Nous utilisons une séparation temporelle : données d'entraînement (2003-2022) et test (2023).  
Les variables explicatives sont standardisées sur l'échantillon d'entraînement pour éviter toute fuite d'information.


## 1. Importation des Bibliothèques et Chargement des Données

Nous importons les librairies nécessaires et chargeons le jeu de données.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


In [2]:
# 1. Chargement et préparation des données
df = pd.read_csv(r"C:\Users\march\OneDrive\Bureau\Data_Science\Dossier\03.code\final_merged_data_normalized.csv", sep=",")  # Adapter le chemin si nécessaire

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
# Vérifiez un aperçu du jeu de données
print(df.head())

        Date             Mine  Production   Latitude  Longitude  Cloud_Cover  \
0 2003-01-01           Andina   19.799999 -33.150873 -70.261021    24.800001   
1 2003-01-01         Salvador    5.700000 -26.282693 -69.597913    47.799999   
2 2003-01-01       Collahuasi   35.799999 -20.982384 -68.632167    74.599998   
3 2003-01-01  Quebrada Blanca    6.300000 -21.004622 -68.799554    74.599998   
4 2003-01-01          El Abra   17.500000 -21.925813 -68.829683    71.700005   

   Diurnal_Temp_Range    Frost_Days  Potential_Evapotranspiration  \
0           11.200000  6.177600e+14                           4.4   
1            8.200000  0.000000e+00                           4.4   
2           10.100000  8.873280e+14                           3.1   
3            9.800000  7.594560e+14                           3.1   
4            9.400001  4.345920e+14                           3.3   

   Cloud_Cover_historique  ...  Anomalie_Temperature_norm  Temp_Min_norm  \
0               31.300001  .

## 2. Définition des Variables et Séparation des Ensembles d'Entraînement et de Test

Les variables explicatives et la variable cible sont extraites, et nous standardisons les features sur l'ensemble d'entraînement.


In [3]:
# Sélection des variables clés
key_vars = ['Production', 'Temperature', 'Precipitation', 'Cloud_Cover', 'Diurnal_Temp_Range', 
            'Frost_Days', 'Potential_Evapotranspiration', 'Anomalie_Cloud_Cover',
            'Anomalie_Diurnal_Temp_Range', 'Anomalie_Frost_Days', 
            'Anomalie_Potential_Evapotranspiration', 'Anomalie_Precipitation', 'Anomalie_Temperature',
            'Temp_Min', 'Temp_Max', 'Vapor_Pressure', 'Wet_Days', 'Anomalie_Vapor_Pressure',
            'Anomalie_Wet_Days', 'prix_lme', 'export_tot']

In [4]:
# On conserve les colonnes d'intérêt
df = df[['Date', 'Mine', 'Year'] + key_vars]

In [5]:
# Définir la variable cible et les features
target_column = "Production"
feature_cols = [col for col in key_vars if col != target_column]

In [6]:
# Split aléatoire 80/20 (reproductible grâce à random_state)
X = df[feature_cols]
y = df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [7]:
# Standardisation des features (fit sur le train uniquement)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [9]:
# Standardisation des features sur l'échantillon d'entraînement pour éviter toute fuite d'information
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Fonction d'affichage des métriques
def print_metrics(y_train, y_pred_train, y_test, y_pred_test, model_name):
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    
    print(f"--- {model_name} ---")
    print(f"Train : R² = {r2_train:.3f}, RMSE = {rmse_train:.3f}, MAE = {mae_train:.3f}")
    print(f"Test  : R² = {r2_test:.3f}, RMSE = {rmse_test:.3f}, MAE = {mae_test:.3f}\n")


## 3. Régression Linéaire OLS

Nous ajustons un modèle de régression linéaire simple (OLS) pour servir de référence.


In [9]:
#Régression linéaire OLS
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr_train = lr.predict(X_train_scaled)
y_pred_lr_test  = lr.predict(X_test_scaled)
print_metrics(y_train, y_pred_lr_train, y_test, y_pred_lr_test, "OLS")


--- OLS ---
Train : R² = 0.030, RMSE = 24.023, MAE = 15.727
Test  : R² = 0.003, RMSE = 25.726, MAE = 16.345



## 4. Régression Ridge avec Validation Croisée

La régression Ridge applique une pénalisation L2 pour réduire le sur-ajustement. Nous utilisons une validation croisée chronologique pour déterminer le meilleur alpha.


In [10]:
# Régression Ridge avec validation croisée

alphas = [0.1, 1.0, 10.0, 50.0]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
y_pred_ridge_train = ridge_cv.predict(X_train_scaled)
y_pred_ridge_test  = ridge_cv.predict(X_test_scaled)
print(f"Meilleur alpha Ridge: {ridge_cv.alpha_:.3f}")
print_metrics(y_train, y_pred_ridge_train, y_test, y_pred_ridge_test, "Ridge")

Meilleur alpha Ridge: 1.000
--- Ridge ---
Train : R² = 0.029, RMSE = 24.032, MAE = 15.727
Test  : R² = 0.004, RMSE = 25.716, MAE = 16.317



## 5. Régression Lasso avec Validation Croisée

La régression Lasso applique une pénalisation L1 qui peut mettre à zéro certains coefficients, facilitant la sélection des variables les plus influentes.


In [11]:
# Régression Lasso avec validation croisée

lasso_cv = LassoCV(cv=5, random_state=0)
lasso_cv.fit(X_train_scaled, y_train)
y_pred_lasso_train = lasso_cv.predict(X_train_scaled)
y_pred_lasso_test  = lasso_cv.predict(X_test_scaled)
print(f"Meilleur alpha Lasso: {lasso_cv.alpha_:.3f}")
print_metrics(y_train, y_pred_lasso_train, y_test, y_pred_lasso_test, "Lasso")

C:\Users\march\anaconda3\envs\python_env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 16459.362695896998, tolerance: 124.11867668690971
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\march\anaconda3\envs\python_env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 173.70945392223075, tolerance: 123.96243129668306
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\march\anaconda3\envs\python_env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 228.13629777962342, tolerance: 123.96243129668306
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\march\anaconda3\envs\python_env

Meilleur alpha Lasso: 0.028
--- Lasso ---
Train : R² = 0.029, RMSE = 24.037, MAE = 15.746
Test  : R² = 0.004, RMSE = 25.717, MAE = 16.310



In [12]:
# Identification des variables sélectionnées par Lasso (coefficients non nuls)
coef_lasso = pd.Series(lasso_cv.coef_, index=feature_cols)
selected_features = coef_lasso[coef_lasso != 0].index.tolist()
print("Variables explicatives sélectionnées par Lasso:", selected_features)

Variables explicatives sélectionnées par Lasso: ['Precipitation', 'Cloud_Cover', 'Frost_Days', 'Potential_Evapotranspiration', 'Anomalie_Cloud_Cover', 'Anomalie_Diurnal_Temp_Range', 'Anomalie_Frost_Days', 'Anomalie_Potential_Evapotranspiration', 'Anomalie_Temperature', 'Temp_Min', 'Temp_Max', 'Wet_Days', 'Anomalie_Vapor_Pressure', 'Anomalie_Wet_Days', 'prix_lme', 'export_tot']


Les modèles linéaires (OLS, Ridge et Lasso) affichent des R² très faibles (autour de 0 sur le test), indiquant qu'ils expliquent pratiquement aucune variance de la production. Les erreurs (RMSE et MAE) restent similaires sur l'entraînement et le test, ce qui signifie qu'il n'y a pas de sur-ajustement, mais également que le modèle manque de pouvoir prédictif. La régularisation (Ridge et Lasso) n'apporte pas d'amélioration notable, suggérant que les variables explicatives actuelles pourraient être peu pertinentes ou que la relation entre ces variables et la production est non linéaire. Il serait judicieux d'explorer d'autres variables, de considérer des transformations ou d'adopter des modèles non linéaires pour mieux capturer les dynamiques du phénomène étudié.